# Apartment Price Baseline Workflow

Clean driver notebook for the thesis baseline on structured apartment-listing data.

Rules:
- Fixed train/test split from the PostgreSQL snapshot exported in `eda.ipynb`
- Model selection happens only on the training split via OOF CV
- The holdout test split is used once for the locked final model
- Text fields remain excluded from the structured baseline
        


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.baseline_workflow import (
    CV,
    DEFAULT_TARGET,
    TEST_PATH,
    TEXT_COLUMNS,
    TRAIN_PATH,
    build_dataset_card,
    build_diagnostics,
    build_final_test_table,
    build_model_catalog,
    fit_model_on_test,
    run_model_cv_comparison,
    run_model_test_comparison,
    save_output_tables,
    select_cleaning_policy,
)
from src.helper import plot_actual_vs_predicted, plot_metric_bars
from src.process import get_pipeline_config

pd.options.display.float_format = "{:,.4f}".format
OUTPUT_DIR = Path("artifacts/baseline")
        


## Dataset Snapshot

Load the fixed snapshot, show the updated baseline roster, and confirm the current preprocessing scope.
        


In [ ]:
dataset_card = build_dataset_card(TRAIN_PATH, TEST_PATH)
model_catalog = build_model_catalog()

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

display(dataset_card)
display(model_catalog)
display(pd.Series(get_pipeline_config(), name="pipeline_config"))
print(f"Excluded text columns: {TEXT_COLUMNS}")
        


## Cleaning Policy Selection

Choose the deployable cleaning policy by cross-validating the XGBoost baseline on the training split only.
        


In [ ]:
policy_results, selected_policy = select_cleaning_policy(
    train_raw,
    target=DEFAULT_TARGET,
    cv=CV,
)

display(policy_results)
print(f"Selected deployable cleaning policy: {selected_policy}")
        


In [ ]:
plot_metric_bars(
    policy_results,
    "cleaning_policy",
    "MedAPE",
    "Cleaning Policy Comparison by OOF CV MedAPE",
)
plot_metric_bars(
    policy_results,
    "cleaning_policy",
    "R2",
    "Cleaning Policy Comparison by OOF CV R2",
    ascending=False,
)
        


## Baseline Model Comparison

Compare the fixed baseline family on the selected cleaning policy: global median, linear and ridge regressions, a decision tree, and XGBoost.
        


In [ ]:
cv_results, locked_model_name, locked_builder = run_model_cv_comparison(
    train_raw,
    cleaning_policy=selected_policy,
    target=DEFAULT_TARGET,
    cv=CV,
)

display(cv_results)
print(f"Locked final model from OOF CV: {locked_model_name}")
        


In [ ]:
plot_metric_bars(
    cv_results,
    "model",
    "MedAPE",
    "Baseline Model Comparison by OOF CV MedAPE",
)
plot_metric_bars(
    cv_results,
    "model",
    "R2",
    "Baseline Model Comparison by OOF CV R2",
    ascending=False,
)
        


## Final Holdout Evaluation

Score the full baseline family on the untouched holdout test split, then inspect the locked model in detail.
        


In [ ]:
test_results = run_model_test_comparison(
    train_raw,
    test_raw,
    cleaning_policy=selected_policy,
    locked_model_name=locked_model_name,
    target=DEFAULT_TARGET,
)
display(test_results)
plot_metric_bars(
    test_results,
    "model",
    "MedAPE",
    "Baseline Model Comparison on Holdout Test by MedAPE",
)
plot_metric_bars(
    test_results,
    "model",
    "R2",
    "Baseline Model Comparison on Holdout Test by R2",
    ascending=False,
)

locked_evaluation = fit_model_on_test(
    train_raw,
    test_raw,
    locked_builder,
    selected_policy,
    target=DEFAULT_TARGET,
)

final_test_table = build_final_test_table(
    model_name=locked_model_name,
    cleaning_policy=selected_policy,
    evaluation=locked_evaluation,
)
display(final_test_table)

(
    evaluation_df,
    region_summary,
    category_summary,
    price_band_summary,
    worst_predictions,
) = build_diagnostics(
    locked_evaluation.cleaned_test_df,
    locked_evaluation.y_test,
    locked_evaluation.predictions,
)

display(region_summary.head(15))
display(category_summary)
display(price_band_summary)
display(worst_predictions.head(20))



In [ ]:
sorted(locked_evaluation.cleaned_test_df["category_sub"].unique())

In [ ]:
sorted(train_raw["category_sub"].unique())


## Visual Diagnostics

Inspect the selected baseline across geography, price bands, and raw prediction fit on the holdout split.
        


In [ ]:
if not region_summary.empty:
    plot_metric_bars(region_summary, "group", "MedAPE", "Regional Test MedAPE",
        ascending=False,
                     top_n=14)
    plot_metric_bars(
        region_summary,
        "group",
        "R2",
        "Regional Test R2",
        ascending=False,
        top_n=14,
    )
if not price_band_summary.empty:
    plot_metric_bars(price_band_summary, "group", "MedAPE", "Price Band Test MedAPE",
        ascending=False
                     )
    plot_metric_bars(
        price_band_summary,
        "group",
        "R2",
        "Price Band Test R2",
        ascending=False,
    )

plot_actual_vs_predicted(
    locked_evaluation.y_test.to_numpy(),
    locked_evaluation.predictions,
    title=f"{locked_model_name}: Actual vs Predicted on Test Set",
)
        


## Save Output Tables

Persist the baseline comparison and diagnostic tables to a dedicated artifact directory.
        


In [ ]:
saved_tables = save_output_tables(
    dataset_card=dataset_card,
    policy_results=policy_results,
    cv_results=cv_results,
    final_test_table=final_test_table,
    test_results=test_results,
    region_summary=region_summary,
    category_summary=category_summary,
    price_band_summary=price_band_summary,
    worst_predictions=worst_predictions,
    output_dir=OUTPUT_DIR,
)

print(f"Saved {len(saved_tables)} tables to {OUTPUT_DIR.resolve()}")
for name in saved_tables:
    print(OUTPUT_DIR / name)
        
